# Sentinel-1 scenes → gamma0 RTC VV/VH on a common grid, via ASF HyP3

Takes any number of Sentinel-1 acquisitions and an AOI, has ASF HyP3 produce
radiometrically terrain-corrected gamma0, then regrids everything onto one shared
grid and writes a GeoTIFF per date with two named bands (`gamma0_VV`,
`gamma0_VH`).

Whole scenes only. **Burst-by-burst processing lives in
`asf_gamma0_burst.ipynb`** — it goes through a different HyP3 job type, with none
of the options available here. Use it when the AOI is small compared with the
250 km swath; stay here otherwise.

**Why HyP3 rather than a local SAR stack.** OTB has no terrain flattening and is
not packaged for Windows; ISCE2/ISCE3 are Linux-only and do not calibrate
radiometrically. HyP3 runs the processing on ASF's servers and returns exactly the
product asked for: gamma0 RTC, VV and VH, GeoTIFF, UTM.

**What "common grid" means here — and what it does not.** The alignment is purely
*geometric*: cell 9 derives one UTM lattice from the AOI alone, and cell 10
resamples every date onto it, so the same array index holds the same patch of
ground in every output file. Two things are deliberately *not* done:

- no **radiometric** coregistration — no cross-calibration, no histogram matching
  between dates. Each gamma0 keeps its own absolute calibration, so a difference
  between two dates is a real change on the ground rather than something a
  normalisation has flattened. That is what a time series needs;
- no **InSAR** coregistration — no phase, no sub-pixel refinement. Whatever
  misregistration HyP3's geocoding left, usually well under a pixel thanks to the
  precise orbits and the Copernicus DEM, stays.

## How to use it

**1. Install the dependencies.**

```
conda install -c conda-forge asf_search hyp3_sdk rasterio geopandas numpy
```

**2. Get a NASA Earthdata Login** (free, <https://urs.earthdata.nasa.gov>) and
store the credentials in a netrc file in your home directory — on Windows,
`C:/Users/<you>/.netrc`:

```
machine urs.earthdata.nasa.gov
login <username>
password <password>
```

`requests` finds that file on its own, by matching the host name. Nothing else to
configure.

**3. Provide the inputs**, in the parameters cell. Only the first two really need
your attention; everything else ships with a working default.

| Parameter | What to give |
| --- | --- |
| `SLC_PATHS` | The acquisitions to process — as many as you like, one or fifty — as local `.SAFE` / `.zip` paths **or** as bare granule names (`S1A_IW_SLC__1SDV_20241018T094629_..._5D7D`). They only say *which* acquisitions to process — ASF works on its own copy, nothing is uploaded. Supplying a local file additionally lets cell 4 check the coverage burst by burst instead of scene by scene. |
| `AOI` | The area of interest in lon/lat (EPSG:4326): inline WKT, or the path to a `.wkt` / `.geojson` file. It drives both the coverage check and the extent of the output rasters. |
| `MODE` | `"slc_scene"` or `"grd"` — see below. Default `"slc_scene"`. |
| `PIXEL_SIZE` | Resolution in metres, for both the processing and the output grid: 10, 20 or 30. |
| `POLARISATIONS` | The bands to produce, in order. Default `["VV", "VH"]`. |
| `DOWNLOAD_DIR`, `OUT_DIR` | Where the raw products and the final GeoTIFFs land. Relative to this notebook by default; point them outside the repository if you can, the products are heavy. Keep them distinct between two runs — cell 10 searches the tree by acquisition timestamp, not by job. |
| `JOB_SUFFIX` | Change it to force a genuinely new submission rather than recovering the previous one — see the comment in cell 3. |

Everything else — the RTC options, the acquisition timestamps — is derived from
these.

**4. Run cells 1 to 5 and read them.** Cell 4 must report `OK` for every
acquisition; cell 5 lists the granules that will actually be submitted. Nothing
has been spent at this stage — both cells only query the free catalogue.

**5. Run cell 6** and compare the credit balance with the number of granules from
cell 5. This is the moment to drop to 20 or 30 m if the budget is tight.

**6. Run cell 7.** For each granule it either recovers the job already filed under
`JOB_NAME` or submits a new one — the only cell that can spend credits. Because
the check is per granule, three things all work: re-running it after a crash or a
restart recovers the batch without paying again, adding acquisitions to
`SLC_PATHS` submits only the new ones, and a failed job is retried. It is also how
you rebuild `batch` before resuming at cell 8.

**7. Run cell 8, then re-run it every few minutes.** It never blocks: it reports
the job statuses and, once none is left pending or running, downloads and
extracts. Processing takes minutes to a few hours, and nothing is lost if you
close the notebook meanwhile — cell 7 rebuilds the batch from `JOB_NAME`.
Downloading is safe to repeat: it costs bandwidth, not credits. Do not postpone
it too long, HyP3 deletes the products after about two weeks.

**8. Run cells 9 to 11.** Purely local. Change `PIXEL_SIZE` or the AOI and replay
them as often as you like, with no reprocessing and no cost.

Results land in `OUT_DIR`: one GeoTIFF per acquisition, bands `gamma0_VV` and
`gamma0_VH`, all sharing the same grid so they stack pixel to pixel.

**After a kernel restart**, run cells 2, 3, 5, 6 and 7 again to rebuild the state,
then carry on at cell 8. Nothing is resubmitted and nothing is paid twice.

## Two modes

Set by `MODE` in the parameters cell. Both go through the same HyP3 job type,
`RTC_GAMMA`, and end on the same output.

| | `"slc_scene"` | `"grd"` |
| --- | --- | --- |
| source product | SLC | GRD |
| jobs | 1 per acquisition | 1 per acquisition |
| source size | heavy | ~8x lighter |
| native resolution | ~5 × 20 m | ~20 m |
| sensible `PIXEL_SIZE` | 10 m | 20 or 30 m |
| granule ids | the product names | looked up by `asf_search` |

**Which one to pick.** `slc_scene` is the default and the most direct: the
granules are simply the product names, with nothing to look up. `grd` starts from
an already detected, multi-looked product — the phase is gone, which rules out any
later interferometry, but that is irrelevant for gamma0 amplitude, and both the
source and the processing are lighter. Its granules have to be found in the
catalogue, since the GRD of an acquisition is a distinct product from its SLC.

## What each cell does

The **ASF** column separates free catalogue queries from the calls that consume
credits, so nothing is spent before everything checkable has been checked.

| # | What it does | ASF |
| --- | --- | --- |
| 1 | This overview. | — |
| 2 | Imports, and puts the sibling `polygon_to_swaths_bursts` folder on `sys.path`. | — |
| 3 | Every parameter: mode, the acquisitions, the AOI, directories, pixel size, job name, RTC options. Reports how many acquisitions were read and their dates. | — |
| 4 | Coverage check: does each acquisition really reach the AOI, and through which swaths and bursts. Falls back to the scene footprint when the product is not a local SLC. | catalogue, free |
| 5 | Builds the list of granules to submit — the product names in `slc_scene`, one catalogue query per acquisition in `grd`. | catalogue, free |
| 6 | Connects to HyP3 through the netrc, prints the credit balance and the real `submit_rtc_job` signature. | yes |
| 7 | For each granule, recovers the job already filed under `JOB_NAME` or submits a new one. **The only cell that can spend credits.** | yes |
| 8 | Reports the job statuses without blocking, and downloads and extracts once they are all done. Re-run it until then. | yes |
| 9 | Computes the common output grid: AOI bounding box in UTM, snapped to the pixel size. | — |
| 10 | Regrids every downloaded raster onto that grid and writes one 2-band GeoTIFF per acquisition. | — |
| 11 | Reports the outputs: grid of each file and share of valid pixels. | — |

In [11]:
# === 2. Imports ===
import inspect
import re
import sys
import zipfile
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import asf_search as asf
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject

# polygon_to_swaths_bursts is a sibling folder: make it importable
TOOLS = Path.cwd().parent / "polygon_to_swaths_bursts"
sys.path.insert(0, str(TOOLS))
from polygon_to_swaths_bursts import get_intersecting_bursts, parse_polygon

In [ ]:
# === 3. Parameters: adapt to your data ===

# "slc_scene" : the whole IW scene, processed from the SLC product
# "grd"       : the whole scene, from the lighter GRD product. No phase, but
#               plenty for gamma0 amplitude, and ~8x lighter to fetch.
# Burst-by-burst processing lives in asf_gamma0_burst.ipynb: it goes through a
# different HyP3 job type with none of these options.
MODE = "slc_scene"

# The acquisitions to process, named by their Sentinel-1 product. Put as many as
# you like — one or fifty: every cell loops over whatever this list holds.
#
# ASF works on its own copy — nothing is uploaded — so each entry may be either
# a local .SAFE / .zip path or just the bare granule name:
#     "S1B_IW_SLC__1SDV_20170804T215105_20170804T215131_006796_00BF5A_B333"
# Only the AOI coverage check in cell 4 opens the local files, and it falls back
# to the ASF catalogue for the entries it cannot find. slc_scene submits these
# names as granules; grd uses their acquisition times to look up the matching
# GRD granules in the catalogue.
SLC_PATHS = [
    r"C:\Users\guigu\Documents\pro_asus\vigisar\data\data_raw\zta8\S1B_IW_SLC__1SDV_20200804T224848_20200804T224915_022779_02B3BE_0770.zip",
]

# Area of interest, lon/lat (EPSG:4326): inline WKT, or a WKT / GeoJSON file path
AOI = "POLYGON ((-63.808599 -24.00883, -62.462762 -24.00883, -62.462762 -25.167657, -63.808599 -25.167657, -63.808599 -24.00883))"

# Where the files land. A relative path resolves against this notebook's folder;
# an absolute one ("C:/data/hyp3") works just as well and is the better habit,
# since these products are heavy and do not belong in the repo. Both directories
# are created on the fly, missing parent folders included.
DOWNLOAD_DIR = "hyp3_downloads/test"   # raw HyP3 products: zips, then extracted
OUT_DIR = "output/test"                # the final 2-band GeoTIFFs

# Resolution in metres, driving both the processing and the output grid. HyP3
# accepts 10, 20 or 30. Two things to weigh: a 10 m SLC scene is close to 4 GB
# per date to download, and a GRD only resolves ~20 m, so 10 m there merely
# oversamples.
PIXEL_SIZE = 10.0
POLARISATIONS = ["VV", "VH"]

# Free-text label stamped on every job, and the only handle to find them again
# later with hyp3.find_jobs(name=...). It is not unique server-side, and cell 7
# relies on it: any job filed under this exact name in the last two weeks is
# recovered instead of being resubmitted.
#
# HENCE: CHANGE JOB_SUFFIX whenever you want a genuinely new submission — a
# different resolution, different RTC options, a clean retry. Keep the same name
# and cell 7 will hand you back the previous jobs, processed with the previous
# parameters, without a word.
JOB_SUFFIX = "v1"
JOB_NAME = f"zta8-{MODE}-{JOB_SUFFIX}"

# gamma0 + power is the standard pair for analysis: keep the linear scale here
# and convert to dB only for display. Both modes of this notebook go through the
# RTC_GAMMA job type, which accepts all of these — cell 6 prints its signature.
RTC_OPTIONS = dict(
    radiometry="gamma0",
    scale="power",
    resolution=int(PIXEL_SIZE),
    dem_name="copernicus",
    speckle_filter=False,   # filtering is a downstream choice, keep raw data
    dem_matching=False,     # can degrade geolocation over flat or wet terrain
)


TIMESTAMP = re.compile(r"\d{8}T\d{6}")


def acquisition_window(product_name):
    """(start, stop) datetimes read from a Sentinel-1 product name.

    Splitting the name on underscores and indexing by position is unreliable:
    the product-type field is four characters wide, so "SLC_" is padded with an
    underscore while "GRDH" is not, and every later field shifts by one. Picking
    out the two YYYYmmddTHHMMSS tokens works whatever the product type.
    """
    stamps = TIMESTAMP.findall(Path(product_name).stem)
    if len(stamps) < 2:
        raise ValueError(
            f"no pair of acquisition times in {Path(product_name).name!r}"
        )
    return (
        datetime.strptime(stamps[0], "%Y%m%dT%H%M%S"),
        datetime.strptime(stamps[1], "%Y%m%dT%H%M%S"),
    )


# --- How a downloaded raster is paired back with its acquisition --------------
#
# HyP3 does not keep the name of what you submitted. Compare an input granule
# with the archive that comes back for it:
#
#     submitted  S1A_IW_SLC__1SDV_20241018T094629_20241018T094656_056155_06DF67_5D7D
#     returned   S1A_IW_20241018T094629_DVP_RTC10_G_gpuned_5FF0
#
# Product type, resolution and even the trailing hex all change — 5FF0 is not
# 5D7D — so neither the granule name nor the job id can be recognised in the file
# name. The one token that survives the renaming is the ACQUISITION START TIME.
#
# That is what cell 10 matches on, and here it can be the FULL timestamp: both
# modes work at scene level, so the file carries the scene's own start time. Two
# passes on the same day, or two consecutive slices of one pass, therefore stay
# properly separated. (The burst notebook has to fall back to the date alone,
# each burst carrying its own time a few seconds into the scene.)
#
# The failure mode worth remembering: two acquisitions sharing a key get merged
# into one output by cell 10, without any error. The check at the end of this
# cell is there to catch exactly that.
STAMP = "%Y%m%dT%H%M%S"

# Parsing the names here turns a malformed entry into an immediate error, before
# anything is submitted or paid for
STARTS = [acquisition_window(p)[0] for p in SLC_PATHS]
KEYS = [start.strftime(STAMP) for start in STARTS]

print(f"{len(SLC_PATHS)} acquisition(s), {MODE} mode at {PIXEL_SIZE:g} m")
print("dates:", ", ".join(sorted(start.strftime("%Y-%m-%d") for start in STARTS)))
print("job name:", JOB_NAME)

if len(set(KEYS)) != len(KEYS):
    print("WARNING: several acquisitions share their timestamp — cell 10 would "
          "merge them into one output")

In [36]:
# === 4. Does the AOI really fall inside each acquisition? ===
# In slc_scene mode this is the only guard against paying to process a product
# that misses the AOI. In slc_burst and grd modes the catalogue query of cell 5
# selects by intersection anyway, so this is a cross-check.
#
# The burst-by-burst check needs a local SLC, since bursts only exist there — a
# GRD is already debursted and declares an empty burst list. Everything else, a
# GRD or an entry given as a bare granule name, falls back to the scene footprint
# from the ASF catalogue: a free query, no download, which also confirms that the
# granule name exists.
aoi_geom = parse_polygon(AOI)

for slc in SLC_PATHS:
    name = Path(slc).stem

    if Path(slc).exists() and "_SLC" in name:
        _, summary = get_intersecting_bursts(slc, AOI, coarse=True)
        if summary:
            detail = ", ".join(f"{sw} {bursts}" for sw, bursts in sorted(summary.items()))
            print(f"OK      {name[:58]}\n        {detail}")
        else:
            print(f"NO DATA {name[:58]} — the AOI is outside this product")
        continue

    results = asf.granule_search([name])
    if not results:
        print(f"UNKNOWN {name[:58]} — no such granule in the ASF catalogue")
    elif any(parse_polygon(r.geometry).intersects(aoi_geom) for r in results):
        print(f"OK      {name[:58]}\n        AOI inside the scene footprint (catalogue)")
    else:
        print(f"NO DATA {name[:58]} — the AOI is outside this scene")

OK      S1B_IW_SLC__1SDV_20200804T224848_20200804T224915_022779_02
        IW1 [2, 3, 4, 5], IW2 [1, 2, 3, 4, 5, 6, 7, 8, 9], IW3 [1, 2, 3, 4, 5, 6, 7, 8]


c:\Users\guigu\Documents\pro_asus\geo\polygon_to_swaths_bursts\polygon_to_swaths_bursts.py:265: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  aoi = _unwrap_geometry(aoi)


In [ ]:
# === 5. The granules to submit ===
if MODE == "slc_scene":
    # The ASF scene identifier is the product name without its extension
    GRANULES = [Path(p).stem for p in SLC_PATHS]
else:
    # grd mode: the GRD of each acquisition is a distinct product from its SLC,
    # so it has to be looked up. The catalogue is asked by time window and AOI
    # intersection — a one-minute margin absorbs rounding while staying far
    # narrower than the 6 or 12 days between two passes.
    aoi_wkt = parse_polygon(AOI).wkt

    GRANULES = []
    for slc in SLC_PATHS:
        start, stop = acquisition_window(slc)
        results = asf.search(
            platform=asf.PLATFORM.SENTINEL1,
            processingLevel="GRD_HD",
            intersectsWith=aoi_wkt,
            start=start - timedelta(minutes=1),
            end=stop + timedelta(minutes=1),
            polarization=["VV+VH"],   # a GRD scene carries both polarisations
        )
        # sceneName, not fileID: the latter appends "-GRD_HD", which HyP3 rejects
        found = sorted(r.properties["sceneName"] for r in results)
        print(f"{Path(slc).stem[:52]}: {len(found)} GRD granule(s)")
        for granule in found:
            print("   ", granule)
        GRANULES += found

print(f"\n{len(GRANULES)} granules to submit in {MODE} mode")

In [ ]:
# === 6. Connect to HyP3 and check what this version accepts ===
import hyp3_sdk as sdk

print("hyp3_sdk", sdk.__version__)

# Credentials are read from the netrc file in your home directory
# (C:/Users/<you>/.netrc, or _netrc — requests accepts either):
#
#     machine urs.earthdata.nasa.gov
#     login <earthdata username>
#     password <earthdata password>
#
# Called without arguments, HyP3 lets requests pick them up from there. Pass
# prompt="password" or prompt="token" instead to be asked interactively.
netrc = next(
    (p for p in (Path.home() / ".netrc", Path.home() / "_netrc") if p.exists()), None
)
if netrc is None:
    raise FileNotFoundError(
        f"no .netrc or _netrc found in {Path.home()} — create one as shown above, "
        'or switch to sdk.HyP3(prompt="password")'
    )
print("credentials from", netrc)

hyp3 = sdk.HyP3()

info = hyp3.my_info()
print("user:", info.get("user_id"), "| remaining credits:", info.get("remaining_credits"))

# Compare RTC_OPTIONS above with the keywords this version really accepts
print("\nsubmit_rtc_job", inspect.signature(hyp3.submit_rtc_job))

In [ ]:
# === 7. Submit the missing jobs, recover the ones already filed ===
# The guard works granule by granule, so all three situations behave sensibly:
# re-running after a crash recovers everything, adding acquisitions to SLC_PATHS
# submits only the new ones, and nothing is ever paid for twice under the same
# JOB_NAME. Failed jobs are ignored, which retries them.
#
# HyP3 keeps job records long after the products expire, so the lookup is limited
# to the last two weeks: an older job cannot be downloaded any more anyway.
recent = datetime.now(timezone.utc) - timedelta(days=14)
existing = hyp3.find_jobs(name=JOB_NAME, start=recent)


def job_granules(job):
    """The granules one job was submitted for."""
    return job.job_parameters.get("granules", [])


wanted = set(GRANULES)
kept = [
    job for job in existing
    if job.status_code != "FAILED" and any(g in wanted for g in job_granules(job))
]
covered = {g for job in kept for g in job_granules(job)}

# Failing loudly here is much cheaper than silently resubmitting everything
if existing and not covered:
    raise RuntimeError(
        f"{len(existing)} job(s) found under {JOB_NAME!r} but their granules could "
        "not be read — refusing to resubmit and risk paying twice. Inspect "
        "existing[0].job_parameters and fix job_granules()."
    )

batch = sdk.Batch(kept)
for granule in GRANULES:
    if granule not in covered:
        batch += hyp3.submit_rtc_job(granule, name=JOB_NAME, **RTC_OPTIONS)

print(f"{JOB_NAME}: {len(kept)} recovered, {len(batch) - len(kept)} submitted")
for job in batch:
    print(f"    {job.status_code:9s} {job.job_id}")

In [29]:
# === 8. Check the jobs, and download once they are done ===
# Deliberately NOT blocking. hyp3.watch() would hold the kernel for hours, and
# interrupting a blocked kernel is what kills it. Re-run this cell every few
# minutes instead: it reports the statuses, and downloads as soon as all the
# jobs have left PENDING and RUNNING.
#
# HyP3 publishes no progress percentage — only PENDING (queued), RUNNING
# (a worker has it) and SUCCEEDED / FAILED. The elapsed time below is the only
# usable proxy: a whole-scene RTC usually lands within the hour.
batch = hyp3.refresh(batch)
now = datetime.now(timezone.utc)

print(" | ".join(f"{code}: {n}"
                 for code, n in sorted(Counter(j.status_code for j in batch).items())))
for job in batch:
    requested = getattr(job, "request_time", None)
    age = f"{(now - requested).total_seconds() / 60:4.0f} min" if requested else "  ? min"
    print(f"    {job.status_code:9s} {age}  {job.job_id}")

waiting = [job for job in batch if job.status_code in ("PENDING", "RUNNING")]
if waiting:
    print(f"\n{len(waiting)} job(s) still going — re-run this cell in a few minutes")
else:
    failed = [job for job in batch if job.status_code == "FAILED"]
    if failed:
        print(f"\n{len(failed)} job(s) FAILED — re-run cell 7 to retry them")

    succeeded = sdk.Batch([j for j in batch if j.status_code == "SUCCEEDED"])
    download_dir = Path(DOWNLOAD_DIR)
    download_dir.mkdir(parents=True, exist_ok=True)
    zips = succeeded.download_files(location=download_dir)

    for archive in zips:
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(download_dir)
        print("extracted:", Path(archive).name)

SUCCEEDED: 2
    SUCCEEDED  220 min  c7da719d-6bba-4f0c-82c5-8a34f4e4d27e
    SUCCEEDED  220 min  b9d36026-d073-48db-956f-45d25318a9e8


S1A_IW_20220118T091611_DVP_RTC10_G_gpuned_5AEB.zip: 100%|██████████| 3.78G/3.78G [02:53<00:00, 23.4MB/s]
S1A_IW_20220903T091619_DVP_RTC10_G_gpuned_FA32.zip: 100%|██████████| 3.80G/3.80G [03:20<00:00, 20.4MB/s]
100%|██████████| 2/2 [06:15<00:00, 187.63s/it]


extracted: S1A_IW_20220118T091611_DVP_RTC10_G_gpuned_5AEB.zip
extracted: S1A_IW_20220903T091619_DVP_RTC10_G_gpuned_FA32.zip


In [31]:
# === 9. The common output grid: AOI bbox in UTM, snapped to PIXEL_SIZE ===
aoi_gs = gpd.GeoSeries([parse_polygon(AOI)], crs="EPSG:4326")
UTM_CRS = aoi_gs.estimate_utm_crs()
minx, miny, maxx, maxy = aoi_gs.to_crs(UTM_CRS).total_bounds

# The grid comes from the AOI alone, never from the images: that is what makes
# every date land on exactly the same pixel centres.
#
# Snapping the origin to a round multiple of PIXEL_SIZE is a separate matter. It
# turns the grid into a canonical lattice attached to (CRS, pixel size) rather
# than to this particular AOI, which buys three things:
#   - editing the AOI later moves the extent by whole pixels instead of sliding
#     the lattice, so new outputs still stack on the old ones;
#   - neighbouring AOIs snapped the same way share the lattice and mosaic
#     without resampling;
#   - other products already on round grids — Sentinel-2 tiles at 10 m in UTM,
#     notably — overlay pixel to pixel, which matters for radar/optical fusion.
ULX = np.floor(minx / PIXEL_SIZE) * PIXEL_SIZE
ULY = np.ceil(maxy / PIXEL_SIZE) * PIXEL_SIZE
SIZE_X = int(np.ceil((maxx - ULX) / PIXEL_SIZE))
SIZE_Y = int(np.ceil((ULY - miny) / PIXEL_SIZE))
TRANSFORM = from_origin(ULX, ULY, PIXEL_SIZE, PIXEL_SIZE)

print(f"{UTM_CRS.name} (EPSG:{UTM_CRS.to_epsg()})")
print(f"{SIZE_X} x {SIZE_Y} px at {PIXEL_SIZE} m, upper-left ({ULX}, {ULY})")

WGS 84 / UTM zone 21S (EPSG:32721)
3307 x 2716 px at 10.0 m, upper-left (666490.0, 9169790.0)


In [ ]:
# === 10. Regrid and write one 2-band GeoTIFF per acquisition ===
def find_rtc_bands(key, pol, search_dir):
    """The RTC GeoTIFF of one acquisition and polarisation.

    HyP3 renames its outputs, so files are paired with their acquisition through
    the timestamp it does keep — see STAMP in cell 3. Both modes work at scene
    level, so a single file is expected; merge_on_grid still handles several,
    which is what happens when an acquisition spans two consecutive slices.
    """
    matches = [
        p for p in Path(search_dir).rglob(f"*_{pol}.tif")
        if key in p.name and "rgb" not in p.name.lower()
    ]
    if not matches:
        raise FileNotFoundError(f"no {pol} RTC file for {key} under {search_dir}")
    return sorted(matches)


def on_common_grid(src_path):
    """Resample one RTC GeoTIFF onto the shared grid. Nodata becomes NaN."""
    target = np.full((SIZE_Y, SIZE_X), np.nan, dtype="float32")
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=target,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=TRANSFORM,
            dst_crs=UTM_CRS,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return target


def merge_on_grid(paths):
    """Regrid several tiles and merge them: average where they overlap."""
    total = np.zeros((SIZE_Y, SIZE_X), dtype="float64")
    count = np.zeros((SIZE_Y, SIZE_X), dtype="uint16")
    for path in paths:
        values = on_common_grid(path)
        valid = np.isfinite(values) & (values > 0)
        total[valid] += values[valid]
        count[valid] += 1
    merged = np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)
    return merged.astype("float32")


outputs = []
for slc, key in zip(SLC_PATHS, KEYS):
    bands = []
    for pol in POLARISATIONS:
        tiles = find_rtc_bands(key, pol, DOWNLOAD_DIR)
        print(f"{key} {pol}: {len(tiles)} tile(s)")
        bands.append(merge_on_grid(tiles))

    out_path = Path(OUT_DIR) / f"{Path(slc).stem}_gamma0.tif"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=SIZE_Y, width=SIZE_X, count=len(POLARISATIONS),
        dtype="float32", crs=UTM_CRS, transform=TRANSFORM, nodata=np.nan,
        compress="deflate", tiled=True,
    ) as dst:
        for index, (pol, band) in enumerate(zip(POLARISATIONS, bands), start=1):
            dst.write(band, index)
            dst.set_band_description(index, f"gamma0_{pol}")

    print("written:", out_path)
    outputs.append(out_path)

In [ ]:
# === 11. Check the outputs: same grid everywhere, enough valid pixels ===
for path in outputs:
    with rasterio.open(path) as src:
        print(path.name)
        print(f"    {src.crs}, {src.width}x{src.height} px, "
              f"pixel {src.transform.a:g} m")
        for index in range(1, src.count + 1):
            data = src.read(index)
            valid = np.isfinite(data) & (data > 0)
            print(f"    {src.descriptions[index - 1]}: "
                  f"{valid.sum() / data.size:.0%} valid pixels")

S1A_IW_GRDH_1SDV_20220118T091611_20220118T091636_041513_04EFD0_4362_gamma0.tif
    EPSG:32721, 3307x2716 px, pixel 10 m
    gamma0_VV: 100% valid pixels
    gamma0_VH: 100% valid pixels
S1A_IW_GRDH_1SDV_20220903T091619_20220903T091644_044838_055AE0_F3CB_gamma0.tif
    EPSG:32721, 3307x2716 px, pixel 10 m
    gamma0_VV: 100% valid pixels
    gamma0_VH: 100% valid pixels
